# 零、总述

**文本嵌入模型**是强大的 NLP 应用的基石，它们为许多文本生成模型赋能。在本章中，我们将会讨论创建和微调嵌入模型的几种方式，以增加它们的表示和语义能力。

我们从探索嵌入模型是什么开始，之后探讨它们是如何工作的？

# 一、Embedding Models

模型非常难处理非结构化的文本数据，所以，我们可以将文本数据转换为数字的表示，我们称其为 **embedding**。做这种嵌入操作的模型叫做嵌入式模型（Embedding Model）。然而，嵌入向量表示成什么最精确呢？一般情况下，我们可以捕捉文档的语义，也就是那些语义相近的文档应该在其嵌入空间中有着更近的距离或者更高的相似度。但是，文档一开始嵌入到向量空间中是随机分布的，我们就需要训练此模型以让他表现的更优。

有许多训练、微调和引导嵌入模型的方式，但最强大的且最广泛使用的技术是**对比学习（Contrastive Learning）**

# 二、什么是对比学习？

**对比学习**的目标是训练嵌入模型在嵌入的向量空间中让相似文档离得更近，不相似的文档离得更远。训练过程如下图所示：

<center>
<img src="./resources/contrastive_learning.png">
</center>

从**解释的本质**视角，我们可以审视一下对比学习。举个例子：一则故事是这样说的，记者问抢劫犯：“你为什么要抢劫银行？”，他说：“因为银行里面钱多”。虽然说这个答案是正确的，但是这个问题的意图不仅仅是说你为什么要抢劫银行，而是说你为什么要抢劫银行而不是去遵守法律。这就称作**Contrastive explanation**，即：**与 “为什么 P？” 相对的备选项是 “为什么 P 而不是 Q？”**

从不同角度理解问题，或者深入的理解问题所询问的本质，这种思想也可以应用于对比学习，在模型学习文档相似/不相似的时候，它可以更进一步学习为什么这些文档相似/不相似。举个例子：你可以让模型理解一张狗的图像：它有鼻子，尾巴，四条腿。但是有鼻子，尾巴和四条腿的生物不能是只猫么？为了加强模型的这种理解，我们可以问模型“为什么是狗而不是猫？”，那么模型就会对比两个概念，去找它们的差异性。

下图是马和斑马的特征比较：
<center>
<img src="./resources/horse_zebra_contrastive.png">
</center>



# 三、SBERT

`sentence-transformers` 是一个在 NLP 领域内很好的三方库，它解决了 BERT 中实现句子嵌入的问题 —— 计算开销（Computation Overhead）太大了。在 `sentence-transformers` 之前，BERT 中的句子嵌入使用的是一种 Cross-Encoder 的架构。

Cross-Encoder 会让两个句子传递到 Transformer 中，用来预测两个句子的相似程度。它用添加分类头的方式来实现，分类头可以输出相似度分数。然而，当大量的句子进入此架构时，计算量会急剧增加，例如，有 10,000 个句子，那它就需要比较 $n(n-1)/2 = 49,995,000$ 次。另外，Cross-Encoder 不生成嵌入，输出的仅仅是句子之间的相似度分数。

Cross-Encoder 的架构如下图所示：

<center>
<img src="./resources/cross_encoder_architecture.png">
</center>

`sentence-transfomers` 的作者则另辟蹊径，致力于寻找一种计算效率高、且能够生成支持语义相似度比较的 Embedding 的方法。
和 Cross-Encoder 架构不同，`sentence-transformers` 删除掉了分类头，取而代之的是**平均池化（Mean Pooling）**，它用来在最终的输出层上生成嵌入向量。池化层会平均词嵌入向量并且会返回固定维度的向量。

`sentence-transformers` 使用了一个孪生的架构（Siamese Architecture）, 如下图所示。两个等同的 BERT 模型共享了相同的权重和神经网络架构。这些模型的输入是句子，输出是池化后的嵌入向量，模型优化的指标是句子嵌入向量之间的相似度。

<center>
<img src="./resources/sentence_transformers_architecture.png">
</center>


# 四、创建一个嵌入模型

创建嵌入模型有很多种方式，**对比学习（Contrastive Learning）** 是最重要的方式，因为它可以学习有效的语义表示。

那么，使用对比学习创建 Embedding 模型的过程分为三步：

1. 构建对比样本
2. 训练模型
3. 评估模型

## 4.1 构建对比样本

在预训练 Embedding 模型的时候，你经常看到他们使用 NLI（Natural Language Inference, 自然语言推理） 数据集。**NLI** 是这样一项任务：对于给定的前提，判断这个前提是否包含这个假设**(Entailment，包含)**，是否与之冲突**(Contradiction， 冲突)**，或者两者皆不**（Neutral，中立）**。

举个例子：

在前提是："他在电影院看 Coco" 的情况下，假设 “他在家里正在观看 Frozen” 就与它冲突，假设“它在电影院看迪士尼的电影 Coco” 就与之是包含关系。如下图所示

<center>
<img src="./resources/contrastive_samples.png">
<center>

`Entailment` 和 `Contradiction` 都是在描述两个输入的相似程度，正因为这种特性，我们可以使用 NLI 数据集来生成正负样本对。在创建和微调嵌入模型过程中，我们使用的数据源于 **GLUE**（General Language Understanding Evaluation benchmark, 通用语言理解评测基准），GLUE 基准测试包含九种语言的理解任务，用于评估和分析模型性能。

这些任务其中之一是多体裁自然语料推理 (MNLI, Multi-Genre Natural Language Inference) 语料库, 它包含了 392702 条语句对，每一对都被标记为 `Entailment`、`Contradiction`或`Neutral`。由于时间语设备的限制，我们仅使用其中的 50,000 对来训练模型。要注意的是：**数据集越小，训练和微调嵌入模型会变得越不稳定。**

In [ ]:
from datasets import load_dataset

# 从 GLUE 中加载 MNLI 数据集
# 0 - entailment, 1 - neutral, 2 - contradiction

train_dataset = load_dataset(
    "glue", "mnli", split="train"
).select(range(50_000))

train_dataset = train_dataset.remove_columns("idx")

我们看一个例子

In [ ]:
train_dataset[2]

## 4.2 训练嵌入模型

现在我们已经有训练集的数据了，接下来我们来训练嵌入模型。现在让我们从零训练嵌入模型。

那么，我们需要考虑以下几件事

1. 选什么模型作为基础模型呢？
2. 应该使用什么损失函数？
3. 怎么去评估模型的好坏呢？

### 4.2.1 基础模型的选择

训练一个 Embedding 模型，首先需要选择合适的预训练模型作为底座（Backbone），其作用是**将输入文本编码为具有上下文语义信息的嵌入向量（Token 维度），为后续构建句子的向量提供基础**。例如 BERT、RoBERT 和 MPNet 都可以作为 Embeding 模型的底座。

目前，我们可以从两个维度来选择 Embedding 基座：**按模型架构选择** 和 **按预训练模型的来源选择**。除此之外，底座选选择的关键因素还有**目标语言、模型参数规模，最长输入长度和训练与推理成本**等等


#### 4.2.1.1 按模型架构选择

现在语言模型的架构分为以下三种

| 架构           | 代表模型 | 适用场景 |
|--------------|---|---|
| Encoder-only | BERT、RoBERTa、MPNet | 传统文本 Embedding、语义检索 |
| Decoder-only |Qwen、Llama | 基于大语言模型构建 Embedding |
| Encoder-Decoder | T5、BART | 特定文本表示与多任务学习场景 |

因此，对于传统的双编码器 Embedding 模型，优先考虑 Encoder-only 架构的模型，常见选择如下表所示

| 架构          | 代表模型            | 特点            |
|-------------|-----------------|---------------|
| BERT        | BERT-base       | 结构经典，资料丰富     |
| RoBERTa     | RoBERTa-base    | 改进 BERT 预训练方式 |
| MPNet       | MPNet-base      | 结合掩码与排列预测思想   |
| DistillBERT | DistillBERT     | 模型小，训练成本低     |
| ModernBERT  | ModernBERT-base | 较新的编码器架构，支持较长上下文 |

#### 4.2.1.2 按预训练模型的来源选择

目前，我将预训练模型的来源分为以下三种

**1. 通用预训练语言模型（General Pretrained Language Model）**

直接选择已经完成语言预训练的 Transformer，之后再通过对比学习训练 Embedding

代表模型有：
- google-bert/bert-base-uncased
- FacebookAI/roberta-base
- microsoft/mpnet-base
- hfl/chinese-macbert-base

**适合**：从头学习 SBERT、研究不同损失函数、构建自己的句子嵌入模型

**2. 领域预训练语言模型（Domain-Specific Pretrained Language Model）**

选择已经针对某个专业领域进行过预训练的模型，再训练 Embedding

例如：
- BioBERT: 生物医学领域
- SciBERT: 科学文献领域
- ClinicalBERT: 临床文本领域

其优势是：**底座已经学习了一定的专业术语和领域语言特征**

**适合**：专业领域的 Embedding 训练

**3. 已训练好的 Embedding 模型（Pretrained Embedding Model）**

直接选择一个已经通过语义对比学习训练的 Embedding 模型，再用自己的数据进行微调

代表模型：
- BAAI/bge-small-zh-v1.5
- thenlper/gte-small
- sentence-transformers/all-mpnet-base-v2
- intfloat/multilingual-e5-base

这类模型已经具备句子语义表示能力，可以在现有基础上进行领域适配

**适合**：企业 RAG、知识库检索、语义检索等实际项目

由于我们要进行后续的训练和调优，我们使用 `bert-base-uncased` 作为基础的模型（Backbone）

In [ ]:
from sentence_transformers import SentenceTransformer

backbone_model = SentenceTransformer("bert-base-uncased")

### 4.2.2 嵌入模型的损失函数

按照训练目标，可以将常见损失函数划分为以下几类

| **类别**    | **代表损失函数** | **核心目标** |
|-----------|---|---|
| 分类损失      | SoftmaxLoss | 通过分类任务学习语义表示 |
| 相似度回归损失   | CosineSimilarityLoss | 预测句子之间的相似度 |
| 对比损失      | ContrasitiveLoss、OnlineContrastiveLoss | 拉进正样本，推远负样本 |
| 三元组损失     | TripletLoss、BatchHardTripletLoss | 保证正样本比负样本更接近 Anchor |
| 多负样本排序损失  | MultipleNegativesRankingLoss、CachedMultipleNegativesRankingLoss | 从多个候选中识别正确匹配 |
| 相似度排序损失   | CoSENTLoss、AnglEloss | 保持预测的相似度排序与标注一致 |
| 无监督表示学习损失 | ContrastiveTensionLoss、DenoisingAutoEncoderLoss | 从无标签文本学习表示 |
| 蒸馏损失      | MSELoss、MarginMSELoss、DistillKLDivLoss | 将教室模型的知识迁移给学生模型 |
| 结构增强损失 | MatrypshkaLoss、AdaptiveLayerLoss | 支持多维度向量或不同层数的推理 |


下面我们分别从基本原理、数学公式和适用场景来逐个介绍上表中的损失函数

#### 4.2.2.1 分类损失: SoftmaxLoss

SoftmaxLoss 即我们常说的交叉熵损失，它基于分类的文本。

其公式如下：

$$
\mathcal{L}_{CE} = - \sum_{k=1}^K y_k \log(p_k)
$$

其中：
- $K$: 类别数量，NLI 上通常为 3
- $y$: 真实类别的标签分布
- $p$: 模型预测的类别概率

其适用场景有：

- NLI 自然语言推断训练
- 具有明确类别标签的句子对训练
- 复现原始 SBERT 的训练方法


#### 4.2.2.2 相似度回归损失

CoSENTLoss、AnglELoss

假设现在有两个句子$u$, $v$，余弦相似度的数学公式如下

$$
\text{sim}(u, v) = \frac{u v}{||u||||v||}
$$

最后默认均方误差作为损失函数

$$
\mathcal{L}_cos =\frac{1}{N} \sum_{i=1}^N(\text{sim}(u, v) - y_i)^2
$$

其中：
- $N$: 句子对的数量
- $y_i$: 代表第 $i$ 个句子的人工标注的真实相似度

适用场景有：

- Semantic Textual Similarity(STS) 任务
- 有连续相似度标签的数据集
- 希望模型的相似度分数接近标注值的任务

#### 4.2.2.3 对比损失

1. 基本原理
2. 数学公式
3. 使用场景

#### 4.2.2.4 三元组损失

1. 基本原理
2. 数学公式
3. 使用场景

#### 4.2.2.5 多负样本排序损失（MNRL）

1. 基本原理
2. 数学公式
3. 使用场景

#### 4.2.2.6 相似度排序损失

1. 基本原理
2. 数学公式
3. 使用场景

#### 4.2.2.7 无监督表示学习损失

1. 基本原理
2. 数学公式
3. 使用场景

#### 4.2.2.8 蒸馏损失

1. 基本原理
2. 数学公式
3. 使用场景

#### 4.2.2.9 结构增强损失

1. 基本原理
2. 数学公式
3. 使用场景

### 4.2.3 评估指标

# 五、微调嵌入模型

## 5.1 有监督

## 5.2 增强的 SBERT

# 六、无监督学习

## 6.1 基于 Transformer 的序列去噪自编码器

## 6.2 对于 Domain Adaption 使用 TSDAE